In [25]:
import pandas as pd
import numpy as np
import pathlib as pl

cfg_nb = pl.Path("../../load-config.ipynb").resolve(strict=True)
%run $cfg_nb

source_url = "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/integrated_call_samples_v3.20200731.ALL.ped"

sample_set = pd.read_csv(source_url, sep="\t", header=0)

column_renamer = lambda c: c.lower().replace(" ", "_")

sample_set = sample_set.rename(column_renamer, axis=1, inplace=False)
sample_sex = sample_set["gender"].replace({1: "male", 2: "female"}, inplace=False)
sample_set.insert(5, "karyotype", sample_sex)

# insert missing samples
add_samples = [
    {
        "individual_id": "HG02109",
        "family_id": "BB18",
        "population": "ACB",
        "relationship": "child",
        "gender": 2,
        "karyotype": "female",
        "paternal_id": "HG02107",
        "maternal_id": "HG02108"
    },
    {
        "individual_id": "NA24385",
        "family_id": "3140",
        "population": "ASK",
        "relationship": "child",
        "gender": 1,
        "karyotype": "male",
        "paternal_id": "NA24149",
        "maternal_id": "NA24143"
    },
    {
        "individual_id": "NA24631",
        "family_id": "3150",
        "population": "CHN",  # no label - Asian/Chinese in Coriell
        "relationship": "child",
        "gender": 1,
        "karyotype": "male",
        "paternal_id": "NA24694",
        "maternal_id": "NA24695"
    },
]

add_samples = pd.DataFrame.from_records(add_samples)

column_datatypes = sample_set.dtypes

for column, dtype in column_datatypes.items():
    if column in add_samples.columns:
        continue
    if np.issubdtype(dtype, np.integer):
        default_value = 0
    else:
        default_value = "0"
    add_samples.insert(add_samples.shape[1], column, default_value)

sample_set = pd.concat([sample_set, add_samples], axis=0, ignore_index=False)
sample_set.sort_values("individual_id", inplace=True)
sample_set.reset_index(drop=True, inplace=True)

out_annotation = pl.Path(CONFIG["project_repo"]).joinpath("annotation", "norm", "samples_1kg_pedigree.ext.tsv").resolve()
out_annotation.parent.mkdir(exist_ok=True, parents=True)
with open(out_annotation, "w") as table:
    _ = table.write(f"# {TIMESTAMP}\n")
    sample_set.to_csv(table, sep="\t", header=True, index=False)